# Astronomy data primer

**Goal:** understand what is inside a ZTF image well enough to clean it, and check the physics with our own measurements.

Read `docs/astronomy_data_primer.md` alongside this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sep
from astropy.time import Time
from astropy.wcs import WCS

from asthunt.config.load_config import load_config
from asthunt.clean.read_fits_image import read_fits_image
from asthunt.clean.build_bad_pixel_mask import build_bad_pixel_mask
from asthunt.clean.estimate_background import estimate_background

config = load_config("project")
fits_files = sorted(config["paths"]["raw"].glob("ztf_test/sci/**/*sciimg.fits"))
image, header = read_fits_image(fits_files[0])
print(fits_files[0].name, image.shape)

## 1. The header: the image's passport

In [ ]:
explanations = {
    "OBSJD":    "when the shutter opened (Julian Date)",
    "EXPTIME":  "exposure length, seconds",
    "FILTER":   "which colour filter",
    "GAIN":     "electrons per count (DN)",
    "READNOI":  "read noise, electrons",
    "SATURATE": "counts where the detector is full",
    "MAGZP":    "magnitude zeropoint: converts counts to magnitudes",
    "SEEING":   "star blur (FWHM) in PIXELS (read the header comment!)",
    "PIXSCALE": "arcsec per pixel",
    "MAGLIM":   "faintest magnitude reliably detected",
}
for key, meaning in explanations.items():
    print(f"{key:9s} = {str(header[key]):>16s}   {meaning}")

print("\nobserved at:", Time(header["OBSJD"], format="jd").iso, "UTC")

## 2. Pixel values
Most pixels are **sky** (~460 counts). A few are **stars**, up to 65,000. A log-scaled histogram shows both.

In [ ]:
plt.hist(image.ravel(), bins=np.logspace(2, 5, 200))
plt.xscale("log"); plt.yscale("log")
plt.axvline(header["SATURATE"], c="r", ls="--", label="saturation")
plt.xlabel("pixel value (counts)"); plt.ylabel("number of pixels"); plt.legend(); plt.show()
print("median (sky) =", round(np.median(image), 1), "counts")

## 3. Bad pixels
Saturated pixels have lost information: the true brightness could be anything above the limit.

In [ ]:
bad = build_bad_pixel_mask(image, saturation_level=header["SATURATE"])
print("saturated pixels:", int((image >= header["SATURATE"]).sum()))
print("bad after growing a 2-px border:", int(bad.sum()), f"({bad.mean():.3%} of the image)")

y, x = np.unravel_index(np.argmax(image), image.shape)        # the brightest pixel
cut = (slice(y - 25, y + 25), slice(x - 25, x + 25))
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(image[cut], origin="lower", cmap="gray", vmax=header["SATURATE"]); axes[0].set_title("brightest star")
axes[1].imshow(bad[cut], origin="lower", cmap="Reds"); axes[1].set_title("bad pixel mask")
plt.show()

## 4. Background and noise: does the physics hold?
Theory: the sky's noise comes from **photon counting** (Poisson: variance = number of electrons) plus **read noise**.
In counts: `noise = sqrt(sky * GAIN + READNOI**2) / GAIN`.

In [ ]:
background, noise = estimate_background(image, bad)

sky = np.median(background)
predicted_noise = np.sqrt(sky * header["GAIN"] + header["READNOI"] ** 2) / header["GAIN"]
print(f"sky level         : {sky:.1f} counts")
print(f"predicted noise   : {predicted_noise:.2f} counts")
print(f"measured noise    : {np.median(noise):.2f} counts")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(background, origin="lower"); axes[0].set_title("background map (counts)")
axes[1].imshow(noise, origin="lower"); axes[1].set_title("noise map (counts)")
plt.show()

Close, but the measured noise is a bit higher. Real images carry extra noise from flat-fielding, resampling and faint
undetected sources. That's why we **measure** the noise map instead of trusting the formula.

## 5. Magnitudes and the zeropoint
Astronomers measure brightness in **magnitudes**: bigger number = fainter, and 5 magnitudes = 100× fainter.
`mag = MAGZP - 2.5 * log10(total counts)`. So the zeropoint is the magnitude of a source giving exactly 1 count.

In [ ]:
def counts_for_magnitude(mag, zeropoint):
    return 10 ** ((zeropoint - mag) / 2.5)

for mag in [15, 17, 19, 20, 21]:
    print(f"mag {mag}: {counts_for_magnitude(mag, header['MAGZP']):10.0f} total counts")

A mag-19 asteroid gives ~830 counts **spread over many pixels**, on a sky with ~10 counts of noise per pixel.
How detectable is it? The **signal-to-noise ratio** (SNR) within an aperture of N pixels:
`SNR = signal / sqrt(signal / GAIN + N * noise_per_pixel**2)`.

In [ ]:
seeing_pixels = header["SEEING"]                 # already in pixels
aperture_pixels = np.pi * seeing_pixels ** 2               # a circle of radius = FWHM
pixel_noise = np.median(noise)

for mag in [18, 19, 20, 20.5, 21, 21.5]:
    signal = counts_for_magnitude(mag, header["MAGZP"])
    snr = signal / np.sqrt(signal / header["GAIN"] + aperture_pixels * pixel_noise ** 2)
    print(f"mag {mag:4.1f}: SNR = {snr:5.1f}")
print("header MAGLIM =", header["MAGLIM"])

Around the header's `MAGLIM` the SNR falls to about 5: the classic detection limit. **Our ML goal is to recover
real asteroids near and below this limit**, where simple thresholds fail.

## 6. Seeing and the PSF
A star is a point, but the atmosphere blurs it into a blob: the **point spread function** (PSF). Its width at half
height is the **FWHM**, reported as `SEEING` (in pixels, says the header comment). Let's measure it on real stars.

In [ ]:
data = np.ascontiguousarray(image - background, dtype=np.float32)
sources = sep.extract(data, thresh=10, err=noise.astype(np.float32), mask=bad)

# Keep bright but unsaturated stars
stars = sources[(sources["peak"] > 2000) & (sources["peak"] < 20000)]


def measure_fwhm_pixels(star_stamp):
    """FWHM straight from its definition: the width of the region brighter than half the peak.

    Count the pixels above half maximum, treat that area as a circle, return its diameter.
    """
    area = (star_stamp >= star_stamp.max() / 2).sum()
    return 2 * np.sqrt(area / np.pi)


fwhm_values = []
for star in stars:
    cx, cy = int(round(star["x"])), int(round(star["y"]))
    stamp = data[cy - 8: cy + 9, cx - 8: cx + 9]
    if stamp.shape == (17, 17):                 # skip stars at the image edge
        fwhm_values.append(measure_fwhm_pixels(stamp))

print(f"{len(fwhm_values)} stars used")
print(f"measured FWHM: {np.median(fwhm_values):.2f} px   header SEEING: {header['SEEING']:.2f} px")

A close match. **Watch out:** my first attempt used SEP's shape numbers (`a`, `b`, second moments) and got 3.6 px,
65% too big. Second moments weigh every pixel by its distance squared, so the faint outer **wings** of a real PSF
(it's not a perfect Gaussian) inflate them. Measuring the quantity by its definition is often the safest choice.

In [ ]:
star = stars[np.argsort(stars["peak"])[len(stars) // 2]]          # a typical star
cx, cy = int(round(star["x"])), int(round(star["y"]))
stamp = data[cy - 8: cy + 9, cx - 8: cx + 9]

yy, xx = np.indices(stamp.shape)
radius = np.hypot(xx - (star["x"] - cx + 8), yy - (star["y"] - cy + 8))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(stamp, origin="lower", cmap="gray"); axes[0].set_title("one star (17x17 px)")
axes[1].scatter(radius.ravel(), stamp.ravel() / stamp.max(), s=8)
axes[1].axhline(0.5, c="gray", ls="--"); axes[1].set_xlabel("distance from centre (px)"); axes[1].set_ylabel("brightness / peak")
axes[1].set_title("radial profile: half height = FWHM/2")
plt.show()

## 7. WCS: from pixels to sky coordinates
The **World Coordinate System** in the header converts pixel (x, y) into sky position (RA, Dec).
Asteroid catalogues (SkyBoT) and reports to the Minor Planet Center use RA/Dec, never pixels.

In [ ]:
wcs = WCS(header)
centre = wcs.pixel_to_world(image.shape[1] / 2, image.shape[0] / 2)
print("image centre:", centre.to_string("hmsdms"))

corner_a = wcs.pixel_to_world(0, 0)
corner_b = wcs.pixel_to_world(100, 0)
print(f"100 pixels = {corner_a.separation(corner_b).arcsec:.1f} arcsec  (PIXSCALE = {header['PIXSCALE']})")

## Exercises
1. Run section 4 on all 10 images. Which night had the brightest sky? Check `MOONILLF` (moon illumination) in its header.
2. Using the SNR formula, at what magnitude does SNR = 3 on this image? On the image with the worst seeing?
3. A mag-19 asteroid moving at 60 "/hr during a 30 s exposure: how many pixels does it trail? Is that more or less than the FWHM?